Set reproducible CPU-only data pipelines, compute training normalization values, and visualize eight random augmentations beside an original image.

In [ ]:
import json
import random
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src" / "models.py").is_file():
    if PROJECT_ROOT == PROJECT_ROOT.parent:
        raise FileNotFoundError("Could not locate project src/models.py")
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.models import BaselineCNN
from src.train_utils import evaluate, train_one_epoch
from src.transforms import get_transforms
from src.datasets import CsvImageDataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(min(torch.get_num_threads(), 2))
DEVICE = torch.device("cpu")
DATA_ROOT = PROJECT_ROOT / "data" / "catdog"
assert DATA_ROOT.is_dir(), "Run the Day 8 notebook first to export CIFAR-10 images."

stats_dataset = CsvImageDataset(DATA_ROOT / "train.csv", transform=transforms.ToTensor())
stats_loader = DataLoader(stats_dataset, batch_size=256, shuffle=False, num_workers=0)
channel_sum = torch.zeros(3)
channel_squared_sum = torch.zeros(3)
pixel_count = 0
for images, _ in stats_loader:
    channel_sum += images.sum(dim=(0, 2, 3))
    channel_squared_sum += (images ** 2).sum(dim=(0, 2, 3))
    pixel_count += images.size(0) * images.size(2) * images.size(3)
mean = (channel_sum / pixel_count).tolist()
std = torch.sqrt(channel_squared_sum / pixel_count - (channel_sum / pixel_count) ** 2).tolist()
print("Train normalization mean:", mean)
print("Train normalization std:", std)

train_transform = get_transforms(train=True, mean=mean, std=std)
eval_transform = get_transforms(train=False, mean=mean, std=std)
raw_train = ImageFolder(DATA_ROOT / "train")
original, original_label = raw_train[17]
def display_tensor(tensor):
    mean_tensor = torch.tensor(mean).view(3, 1, 1)
    std_tensor = torch.tensor(std).view(3, 1, 1)
    return (tensor * std_tensor + mean_tensor).clamp(0, 1).permute(1, 2, 0).numpy()

fig = plt.figure(figsize=(14, 6))
grid = fig.add_gridspec(2, 5)
original_axis = fig.add_subplot(grid[:, 0])
original_axis.imshow(original)
original_axis.set_title(f"Original: {raw_train.classes[original_label]}")
original_axis.axis("off")
for index in range(8):
    axis = fig.add_subplot(grid[index // 4, index % 4 + 1])
    axis.imshow(display_tensor(train_transform(original)))
    axis.set_title(f"Augmentation {index + 1}")
    axis.axis("off")
fig.tight_layout()
plt.show()

train_eval_dataset = CsvImageDataset(DATA_ROOT / "train.csv", transform=eval_transform)
train_aug_dataset = CsvImageDataset(DATA_ROOT / "train.csv", transform=train_transform)
val_dataset = CsvImageDataset(DATA_ROOT / "val.csv", transform=eval_transform)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=0)
print(f"Train: {len(train_eval_dataset):,}; validation: {len(val_dataset):,}; device: {DEVICE}")


Train the same BaselineCNN twice for 15 epochs with identical initialization seeds, Adam hyperparameters, batch size, and validation data.

In [ ]:
EPOCHS = 15
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
criterion = nn.CrossEntropyLoss()
baseline_histories = {}
baseline_models = {}
baseline_datasets = {
    "BaselineCNN (no aug)": train_eval_dataset,
    "BaselineCNN (aug)": train_aug_dataset,
}

for run_name, training_dataset in baseline_datasets.items():
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    loader_generator = torch.Generator().manual_seed(SEED)
    train_loader = DataLoader(
        training_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        generator=loader_generator,
    )
    model = BaselineCNN().to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    history = {"train_loss": [], "train_acc": [], "val_acc": []}
    for epoch in range(EPOCHS):
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)
        _, val_acc, _ = evaluate(model, val_loader, criterion)
        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)
        print(
            f"{run_name} | epoch {epoch + 1:02d}/{EPOCHS} | "
            f"train loss {train_loss:.4f} | train acc {train_acc:.4f} | val acc {val_acc:.4f}"
        )
    baseline_histories[run_name] = history
    baseline_models[run_name] = model
    print(f"Completed {run_name}.")


Plot train/validation accuracy for both experiments, compute final train-validation gaps, and persist the measured baseline metrics for the Day 10 comparison.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
baseline_metrics = []
for axis, (run_name, history) in zip(axes, baseline_histories.items()):
    epochs = range(1, len(history["train_acc"]) + 1)
    axis.plot(epochs, history["train_acc"], label="Train accuracy")
    axis.plot(epochs, history["val_acc"], label="Validation accuracy")
    axis.set_title(run_name)
    axis.set_xlabel("Epoch")
    axis.set_ylabel("Accuracy")
    axis.grid(alpha=0.3)
    axis.legend()
    best_val_index = int(np.argmax(history["val_acc"]))
    baseline_metrics.append(
        {
            "model": run_name,
            "val_acc": float(history["val_acc"][best_val_index]),
            "test_acc": None,
            "parameters": sum(parameter.numel() for parameter in baseline_models[run_name].parameters()),
            "train_val_gap": float(history["train_acc"][-1] - history["val_acc"][-1]),
        }
    )
fig.tight_layout()
plt.show()

baseline_summary = pd.DataFrame(baseline_metrics)
baseline_summary["train_val_gap"] = baseline_summary["train_val_gap"].map(lambda value: f"{value:.4f}")
display(baseline_summary)
baseline_path = DATA_ROOT / "week2_baseline_metrics.json"
if baseline_path.exists():
    raise FileExistsError(f"Refusing to overwrite existing baseline results: {baseline_path}")
baseline_path.write_text(json.dumps(baseline_metrics, indent=2), encoding="utf-8")
print("Saved baseline results to", baseline_path)
